In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report


try:
    # Using the path provided by the user. Note: For Colab, you need to upload the file or mount Google Drive.
    df = pd.read_csv("/content/z.csv")
    print("Dataset loaded successfully.")
    print(df.head())

    # --- Data Cleaning and Preparation for z.csv ---
    # Convert 'rate' to numeric
    df['rate_numeric'] = df['rate'].apply(lambda x: float(x.split('/')[0]) if isinstance(x, str) and '/' in x and x != 'NEW' else np.nan)
    df['rate_numeric'].fillna(df['rate_numeric'].mean(), inplace=True)

    # Convert 'online_order' to numeric (0/1)
    df['online_order_numeric'] = df['online_order'].map({'Yes': 1, 'No': 0})
    df['online_order_numeric'].fillna(df['online_order_numeric'].mode()[0], inplace=True) # Fill NaNs with mode

    # Convert 'book_table' to numeric (0/1)
    df['book_table_numeric'] = df['book_table'].map({'Yes': 1, 'No': 0})
    df['book_table_numeric'].fillna(df['book_table_numeric'].mode()[0], inplace=True) # Fill NaNs with mode

    # Convert 'approx_cost(for two people)' to numeric
    df['approx_cost_numeric'] = df['approx_cost(for two people)'].astype(str).str.replace(',', '').astype(float)
    df['approx_cost_numeric'].fillna(df['approx_cost_numeric'].mean(), inplace=True)

    # 'votes' column
    df['votes'] = pd.to_numeric(df['votes'], errors='coerce')
    df['votes'].fillna(df['votes'].mean(), inplace=True)

    # Define features (X) and targets (y)
    X = df[['votes', 'rate_numeric', 'online_order_numeric']]
    y_linear = df['approx_cost_numeric']
    y_logistic = df['book_table_numeric'] # Predicting if booking a table is possible

except FileNotFoundError:
    print("Error: The specified dataset file (z.csv) was not found at the provided local path.")
    print("Please ensure 'z.csv' is uploaded to your Colab environment or provide the correct path (e.g., if it's in Google Drive). You can also upload it using the file browser on the left.")
    # Creating a dummy dataset for demonstration if file not found
    print("Creating a dummy dataset for demonstration purposes.")
    data = {
        'feature1': np.random.rand(100) * 10,
        'feature2': np.random.randint(1, 5, 100),
        'target_linear': np.random.rand(100) * 5 + np.random.rand(100) * 2,
        'target_logistic': np.random.randint(0, 2, 100)
    }
    df = pd.DataFrame(data)
    df['target_linear'] = df['feature1'] * 0.5 + df['feature2'] * 1.2 + np.random.randn(100) * 0.5
    df['target_logistic'] = (df['feature1'] + df['feature2'] > 7).astype(int)
    print(df.head())

    # Use dummy data for X and y if file not found
    X = df[['feature1', 'feature2']]
    y_linear = df['target_linear']
    y_logistic = df['target_logistic']

# --- 3. Linear Regression ---
print("\n--- Performing Linear Regression ---")

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X, y_linear, test_size=0.2, random_state=42)

linear_model = LinearRegression()
linear_model.fit(X_train_lr, y_train_lr)

y_pred_lr = linear_model.predict(X_test_lr)

mse = mean_squared_error(y_test_lr, y_pred_lr)
rmse = np.sqrt(mse)
print(f"Mean Squared Error (Linear Regression): {mse:.2f}")
print(f"Root Mean Squared Error (Linear Regression): {rmse:.2f}")
print(f"Linear Regression Coefficients: {linear_model.coef_}")
print(f"Linear Regression Intercept: {linear_model.intercept_:.2f}")

# --- 4. Logistic Regression ---
print("\n--- Performing Logistic Regression ---")


X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X, y_logistic, test_size=0.2, random_state=42)


logistic_model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is good for small datasets
logistic_model.fit(X_train_log, y_train_log)

y_pred_log = logistic_model.predict(X_test_log)
y_pred_proba_log = logistic_model.predict_proba(X_test_log)[:, 1] # Probability of the positive class

accuracy = accuracy_score(y_test_log, y_pred_log)
print(f"Accuracy (Logistic Regression): {accuracy:.2f}")
print("\nClassification Report (Logistic Regression):\n", classification_report(y_test_log, y_pred_log))
print(f"Logistic Regression Coefficients: {logistic_model.coef_}")
print(f"Logistic Regression Intercept: {logistic_model.intercept_[0]:.2f}")

Dataset loaded successfully.
                    name online_order book_table   rate  votes  \
0                  Jalsa          Yes        Yes  4.1/5    775   
1         Spice Elephant          Yes         No  4.1/5    787   
2        San Churro Cafe          Yes         No  3.8/5    918   
3  Addhuri Udupi Bhojana           No         No  3.7/5     88   
4          Grand Village           No         No  3.8/5    166   

   approx_cost(for two people) listed_in(type)  
0                          800          Buffet  
1                          800          Buffet  
2                          800          Buffet  
3                          300          Buffet  
4                          600          Buffet  

--- Performing Linear Regression ---
Mean Squared Error (Linear Regression): 44579.68
Root Mean Squared Error (Linear Regression): 211.14
Linear Regression Coefficients: [5.67349708e-02 3.50605042e+01 1.37035931e+02]
Linear Regression Intercept: 233.98

--- Performing Logistic R

/tmp/ipykernel_15171/2755064417.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['rate_numeric'].fillna(df['rate_numeric'].mean(), inplace=True)
/tmp/ipykernel_15171/2755064417.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value

### Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

print("\n--- Performing Decision Tree Classification ---")

decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train_log, y_train_log)

y_pred_dt = decision_tree_model.predict(X_test_log)

accuracy_dt = accuracy_score(y_test_log, y_pred_dt)
print(f"Accuracy (Decision Tree): {accuracy_dt:.2f}")
print("\nClassification Report (Decision Tree):\n", classification_report(y_test_log, y_pred_dt))


--- Performing Decision Tree Classification ---
Accuracy (Decision Tree): 0.93

Classification Report (Decision Tree):
               precision    recall  f1-score   support

           0       0.96      0.96      0.96        28
           1       0.50      0.50      0.50         2

    accuracy                           0.93        30
   macro avg       0.73      0.73      0.73        30
weighted avg       0.93      0.93      0.93        30



### Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("\n--- Performing Random Forest Classification ---")

random_forest_model = RandomForestClassifier(random_state=42)
random_forest_model.fit(X_train_log, y_train_log)

y_pred_rf = random_forest_model.predict(X_test_log)

accuracy_rf = accuracy_score(y_test_log, y_pred_rf)
print(f"Accuracy (Random Forest): {accuracy_rf:.2f}")
print("\nClassification Report (Random Forest):\n", classification_report(y_test_log, y_pred_rf))


--- Performing Random Forest Classification ---
Accuracy (Random Forest): 0.93

Classification Report (Random Forest):
               precision    recall  f1-score   support

           0       0.93      1.00      0.97        28
           1       0.00      0.00      0.00         2

    accuracy                           0.93        30
   macro avg       0.47      0.50      0.48        30
weighted avg       0.87      0.93      0.90        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Support Vector Machine (SVM) Classifier

In [ ]:
from sklearn.svm import SVC

print("\n--- Performing SVM Classification ---")

# Note: SVM can be sensitive to feature scaling. For simplicity, we'll use it directly.
# For better performance, consider scaling features (e.g., StandardScaler) before applying SVM.
svm_model = SVC(random_state=42)
svm_model.fit(X_train_log, y_train_log)

y_pred_svm = svm_model.predict(X_test_log)

accuracy_svm = accuracy_score(y_test_log, y_pred_svm)
print(f"Accuracy (SVM): {accuracy_svm:.2f}")
print("\nClassification Report (SVM):\n", classification_report(y_test_log, y_pred_svm))


--- Performing SVM Classification ---
Accuracy (SVM): 0.93

Classification Report (SVM):
               precision    recall  f1-score   support

           0       0.93      1.00      0.97        28
           1       0.00      0.00      0.00         2

    accuracy                           0.93        30
   macro avg       0.47      0.50      0.48        30
weighted avg       0.87      0.93      0.90        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Principal Component Analysis (PCA)

PCA is a dimensionality reduction technique. It's often used to reduce the number of features in your dataset while retaining as much variance as possible. This can help with model performance and visualization.

Since the current dataset has only 3 features, PCA might not offer significant dimensionality reduction benefit for *this specific feature set*, but it's a valuable technique to understand. I'll demonstrate it here, and you can apply it in scenarios with many more features.

We will apply PCA and then use the transformed data with a simple classifier (e.g., Decision Tree) to show its usage.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("\n--- Performing PCA ---")

# It's good practice to scale your data before applying PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA to reduce dimensions
# Let's reduce it to 2 components for demonstration purposes
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Original number of features: {X.shape[1]}")
print(f"Reduced number of features (PCA): {X_pca.shape[1]}")
print(f"Explained variance ratio by principal components: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.2f}")

# Optionally, train a classifier on the PCA-transformed data
# We'll use logistic regression as an example
print("\n--- Logistic Regression after PCA ---")

X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y_logistic, test_size=0.2, random_state=42)

logistic_model_pca = LogisticRegression(random_state=42, solver='liblinear')
logistic_model_pca.fit(X_train_pca, y_train_pca)

y_pred_log_pca = logistic_model_pca.predict(X_test_pca)

accuracy_log_pca = accuracy_score(y_test_pca, y_pred_log_pca)
print(f"Accuracy (Logistic Regression with PCA): {accuracy_log_pca:.2f}")
print("\nClassification Report (Logistic Regression with PCA):\n", classification_report(y_test_pca, y_pred_log_pca))


--- Performing PCA ---
Original number of features: 3
Reduced number of features (PCA): 2
Explained variance ratio by principal components: [0.62379629 0.21345599]
Total explained variance: 0.84

--- Logistic Regression after PCA ---
Accuracy (Logistic Regression with PCA): 0.93

Classification Report (Logistic Regression with PCA):
               precision    recall  f1-score   support

           0       0.93      1.00      0.97        28
           1       0.00      0.00      0.00         2

    accuracy                           0.93        30
   macro avg       0.47      0.50      0.48        30
weighted avg       0.87      0.93      0.90        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Long Short-Term Memory (LSTM) with `z.csv`

As discussed previously, `z.csv` contains tabular data, not inherently sequential data. To use an LSTM, we need to reshape the data into a 3D format: `(samples, timesteps, features)`. For this demonstration, we'll treat each row as a single timestep, making the `timesteps` dimension equal to 1. This is a common workaround when applying LSTMs to non-sequential tabular data, although its effectiveness depends heavily on the problem and data.

First, we need to install `tensorflow` and `keras`.

Now, let's train the LSTM model using the prepared data and then evaluate its performance.

In [40]:
# Train the model
history = lstm_model.fit(
    X_train_lstm, y_train_lstm, epochs=50, batch_size=16, validation_split=0.2, verbose=0
)

# Evaluate the model on the test set
y_pred_proba_lstm = lstm_model.predict(X_test_lstm)
y_pred_lstm = (y_pred_proba_lstm > 0.5).astype(int)

accuracy_lstm = accuracy_score(y_test_lstm, y_pred_lstm)
print(f"Accuracy (LSTM): {accuracy_lstm:.2f}")
print("\nClassification Report (LSTM):\n", classification_report(y_test_lstm, y_pred_lstm))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 635ms/step
Accuracy (LSTM): 0.93

Classification Report (LSTM):
               precision    recall  f1-score   support

           0       0.93      1.00      0.97        28
           1       0.00      0.00      0.00         2

    accuracy                           0.93        30
   macro avg       0.47      0.50      0.48        30
weighted avg       0.87      0.93      0.90        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Conceptual Reinforcement Learning (Q-Learning) with `z.csv`

Reinforcement Learning (RL) is designed for agents to learn optimal behavior by interacting with an environment. As `z.csv` is a static, tabular dataset without inherent sequential interaction or clear actions/rewards, directly applying RL is not straightforward.

However, to demonstrate the *structure* of an RL solution, we can create a highly conceptual and artificial environment based on `z.csv` and apply a basic Q-learning algorithm. This will illustrate the components of an RL system, even if the 'learned' policy isn't meaningful for this particular dataset.

We will:
1.  **Discretize States**: Convert continuous features into discrete bins to create a state space.
2.  **Define Arbitrary Actions**: Introduce a fixed set of actions (e.g., 'Action A', 'Action B').
3.  **Assign Artificial Rewards**: Based on the `y_logistic` target, we'll provide rewards.
4.  **Implement Q-Learning**: Apply the Q-learning algorithm to learn a Q-table.

**Note**: This is a purely conceptual exercise to show the RL framework. A genuine RL problem would require a dynamic environment with agent-driven state transitions and rewards that reflect real-world objectives.

In [42]:
import numpy as np
from sklearn.preprocessing import KBinsDiscretizer

print("--- Conceptual Reinforcement Learning with z.csv ---")
print("z.csv is a static tabular dataset, not a typical Reinforcement Learning environment.")
print("To demonstrate RL, we will create a highly conceptual and artificial setup:")
print(" - States will be derived by discretizing the features from X_scaled_lstm.")
print(" - Actions will be arbitrary (e.g., 'Action A', 'Action B').")
print(" - Rewards will be artificially assigned based on y_logistic.")
print(" - 'Next state' will simply be the subsequent entry in the dataset, not a result of action in an environment.")
print("This is purely to illustrate the Q-learning framework, not to solve a meaningful RL problem with this data.")

# --- Discretize States ---
# Use KBinsDiscretizer to convert continuous features into discrete bins
# We'll use 5 bins per feature for demonstration.
# The `X_scaled_lstm` already has shape (samples, features), which is suitable for the discretizer.
X_flat = X_scaled_lstm

discretizer = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform', subsample=None)
X_discretized = discretizer.fit_transform(X_flat).astype(int)

# Map discrete feature combinations to a single state index
unique_states = {}
current_state_idx = 0
state_indices = np.zeros(len(X_discretized), dtype=int)

for i, state_row in enumerate(X_discretized):
    state_tuple = tuple(state_row)
    if state_tuple not in unique_states:
        unique_states[state_tuple] = current_state_idx
        current_state_idx += 1
    state_indices[i] = unique_states[state_tuple]

num_states = current_state_idx # The actual number of unique states observed in the data
print(f"\nOriginal feature space: {X_flat.shape[1]} features")
print(f"Discretized features into {discretizer.n_bins_[0]} bins each.")
print(f"Number of unique discrete states derived: {num_states}")

# --- Define Actions ---
num_actions = 2 # Arbitrary actions, e.g., 'Action A', 'Action B'
print(f"Number of arbitrary actions defined: {num_actions}")

# --- Q-Table Initialization ---
q_table = np.zeros((num_states, num_actions))
print(f"Initialized Q-table of shape: {q_table.shape}")

# --- Hyperparameters ---
learning_rate = 0.8
discount_factor = 0.95
epsilon = 0.1 # Exploration-exploitation trade-off
epochs = 1000 # Number of training episodes

print(f"\nTraining Q-learning agent for {epochs} epochs...")

# --- Training Loop (Conceptual) ---
for epoch in range(epochs):
    # For each episode, we'll iterate through the entire dataset
    for i in range(len(state_indices)):
        current_state_idx_val = state_indices[i]

        # Epsilon-greedy action selection
        if np.random.uniform(0, 1) < epsilon:
            action = np.random.randint(num_actions) # Explore
        else:
            action = np.argmax(q_table[current_state_idx_val, :]) # Exploit

        # --- Define Arbitrary Reward and Next State ---
        # Reward: +1 if y_logistic for this state is 1 (booked table), else 0
        reward = 1 if y_logistic.iloc[i] == 1 else 0

        # Next State (highly artificial): The next state in the sequence of the dataset
        # If it's the last state, loop back to the first state
        next_state_idx_val = state_indices[(i + 1) % len(state_indices)]

        # Q-learning update formula
        old_value = q_table[current_state_idx_val, action]
        next_max = np.max(q_table[next_state_idx_val, :])
        new_value = (1 - learning_rate) * old_value + learning_rate * (reward + discount_factor * next_max)
        q_table[current_state_idx_val, action] = new_value

# --- Evaluation (Conceptual) ---
print("\nQ-learning training complete.")
print("Example Q-table (first 5 states):\n", q_table[:5, :])

print("\nInterpretation: This Q-table now contains 'learned' values for taking 'Action A' or 'Action B' from different conceptual states.")
print("However, because the environment dynamics (next state, reward) were artificially defined based on static data,")
print("these 'learned' policies are unlikely to represent a true RL solution or generalize to a dynamic environment.")

--- Conceptual Reinforcement Learning with z.csv ---
z.csv is a static tabular dataset, not a typical Reinforcement Learning environment.
To demonstrate RL, we will create a highly conceptual and artificial setup:
 - States will be derived by discretizing the features from X_scaled_lstm.
 - Actions will be arbitrary (e.g., 'Action A', 'Action B').
 - Rewards will be artificially assigned based on y_logistic.
 - 'Next state' will simply be the subsequent entry in the dataset, not a result of action in an environment.
This is purely to illustrate the Q-learning framework, not to solve a meaningful RL problem with this data.

Original feature space: 3 features
Discretized features into 5 bins each.
Number of unique discrete states derived: 15
Number of arbitrary actions defined: 2
Initialized Q-table of shape: (15, 2)

Training Q-learning agent for 1000 epochs...

Q-learning training complete.
Example Q-table (first 5 states):
 [[1.04111543 1.30856127]
 [1.02519537 1.07727279]
 [1.078456 

### Reinforcement Learning: CartPole Example (Q-Learning)

This section demonstrates a more traditional Reinforcement Learning setup using the classic **CartPole environment** from OpenAI Gym. The goal is for an agent to learn to balance a pole on a cart by moving the cart left or right. This environment has a clear state space, action space, and reward function.

First, we'll install the `gym` library.

In [48]:
import sys

# Uninstall OpenAI Gym (old version) and install Gymnasium
!{sys.executable} -m pip uninstall -y gym
!{sys.executable} -m pip install gymnasium
!{sys.executable} -m pip install "gymnasium[classic_control]" # For classic control environments like CartPole

Found existing installation: gym 0.25.2
Uninstalling gym-0.25.2:
  Successfully uninstalled gym-0.25.2


Now, let's import the necessary libraries and create the CartPole environment. We'll also define helper functions to discretize the continuous state space, which is necessary for traditional Q-learning.

In [51]:
import numpy as np
import gymnasium as gym # Import gymnasium instead of gym

# Create the CartPole environment
env = gym.make('CartPole-v1')

# Define the number of discrete bins for each observation dimension
# CartPole state space: [cart position, cart velocity, pole angle, pole angular velocity]
pos_space = np.linspace(-2.4, 2.4, 10)  # Position of cart
vel_space = np.linspace(-4, 4, 10)      # Velocity of cart
ang_space = np.linspace(-0.2095, 0.2095, 10) # Angle of pole
ang_vel_space = np.linspace(-4, 4, 10)  # Angular velocity of pole

def get_discrete_state(state):
    """Converts a continuous state observation into a discrete index."""
    pos, vel, ang, ang_vel = state
    pos_idx = np.digitize(pos, pos_space) - 1
    vel_idx = np.digitize(vel, vel_space) - 1
    ang_idx = np.digitize(ang, ang_space) - 1
    ang_vel_idx = np.digitize(ang_vel, ang_vel_space) - 1

    # Ensure indices are within valid range [0, num_bins-1]
    pos_idx = np.clip(pos_idx, 0, len(pos_space) - 1)
    vel_idx = np.clip(vel_idx, 0, len(vel_space) - 1)
    ang_idx = np.clip(ang_idx, 0, len(ang_space) - 1)
    ang_vel_idx = np.clip(ang_vel_idx, 0, len(ang_vel_space) - 1)

    return tuple([pos_idx, vel_idx, ang_idx, ang_vel_idx])

print("CartPole environment created.")
print(f"Observation space shape: {env.observation_space.shape}")
print(f"Action space: {env.action_space}") # 0: left, 1: right

# Calculate the total number of discrete states
num_states_cartpole = (len(pos_space) * len(vel_space) * len(ang_space) * len(ang_vel_space))
print(f"Number of discrete states (CartPole): {num_states_cartpole}")

CartPole environment created.
Observation space shape: (4,)
Action space: Discrete(2)
Number of discrete states (CartPole): 10000


Now we can define the Q-learning algorithm, which will train an agent to make decisions in the CartPole environment. We'll initialize a Q-table and update it over many episodes.

Finally, let's evaluate the performance of our trained Q-learning agent. We'll run a few episodes and observe how well it balances the pole.

In [ ]:
print("\n--- Evaluating Q-learning agent for CartPole ---")

env_eval = gym.make('CartPole-v1')
num_eval_episodes = 10
episode_rewards = []

for episode in range(num_eval_episodes):
    state_continuous, info = env_eval.reset() # env.reset() returns (observation, info) in gymnasium
    discrete_state = get_discrete_state(state_continuous)
    terminated = False # Use 'terminated' for gymnasium
    truncated = False
    total_reward = 0

    for step in range(max_steps_per_episode_cartpole):
        action = np.argmax(q_table_cartpole[discrete_state]) # Always exploit trained policy
        next_state_continuous, reward, terminated, truncated, info = env_eval.step(action) # env.step() returns (observation, reward, terminated, truncated, info)
        discrete_state = get_discrete_state(next_state_continuous)
        total_reward += reward

        if terminated or truncated:
            break

    episode_rewards.append(total_reward)
    print(f"Evaluation Episode {episode + 1}: Total Reward = {total_reward}")

env_eval.close()
print(f"\nAverage reward over {num_eval_episodes} evaluation episodes: {np.mean(episode_rewards):.2f}")
print("\nInterpretation: A higher average reward indicates better performance in balancing the pole. The CartPole-v1 environment is considered 'solved' if an agent gets an average reward of 195.0 over 100 consecutive trials.")

### Long Short-Term Memory (LSTM) Networks

LSTM networks are a type of recurrent neural network (RNN) primarily designed for sequence prediction problems, such as time series forecasting, natural language processing, or speech recognition. They are particularly good at learning long-term dependencies.

Your current dataset `z.csv` appears to be tabular data without an explicit sequential or temporal component that would naturally lend itself to an LSTM model. To use an LSTM, we would typically need to:

1.  **Structure the data as sequences:** This might involve creating sequences from time-series data (e.g., daily sales over several days as input to predict the next day's sales) or transforming other data into sequential form.
2.  **Reshape the input:** LSTM layers expect input data to be in a 3D format: `(samples, timesteps, features)`.

Given the nature of the `z.csv` data (static restaurant information), applying an LSTM directly would require significant feature engineering to create meaningful sequences. If you have a specific sequential aspect of the data in mind (e.g., tracking a restaurant's performance over time), please let me know, and we can explore how to prepare the data for an LSTM model.